> **Disclaimer:** This is a research project. Models are not validated
> for clinical use and must not be used for medical diagnosis or
> treatment decisions. The NIH ChestX-ray14 labels were text-mined from
> radiology reports with an estimated 10–15% label noise.

# ResNet50 Baseline Model Training

**Purpose:** Train ResNet50 as baseline model for chest X-ray disease classification

**Architecture:** ResNet50 with pretrained ImageNet weights, modified for grayscale input

**Dataset:** NIH ChestX-ray14 (112,120 images, 14 disease labels)

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from PIL import Image
import json
from tqdm import tqdm
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
from src.dataset import ChestXrayDataset, apply_clahe, get_transforms


## Configuration

In [ ]:
DATA_ROOT = Path('./data')
PROCESSED_DIR = DATA_ROOT / 'processed'
MODELS_DIR = Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
NUM_WORKERS = 4

## Load Data

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / 'train_df.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_df.csv')

with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    config = json.load(f)

diseases = config['diseases']
NUM_CLASSES = len(diseases)
IMG_SIZE = config['image_size']

print(f"Training samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

## Dataset and DataLoader

In [ ]:
# Moved to src/dataset.py

In [ ]:
train_transform, val_transform = get_transforms(IMG_SIZE)

train_dataset = ChestXrayDataset(train_df, diseases, transform=train_transform)
val_dataset = ChestXrayDataset(val_df, diseases, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

## Model Architecture

In [ ]:
class ResNet50Model(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(ResNet50Model, self).__init__()
        
        if pretrained:
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        else:
            self.backbone = models.resnet50(weights=None)
        
        self.backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

model = ResNet50Model(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Loss Function and Optimizer

In [ ]:
pos_weights = torch.load(PROCESSED_DIR / 'class_weights.pt', map_location=device, weights_only=True)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

## Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Evaluating')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            preds = torch.sigmoid(outputs)
            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)
    
    epoch_loss = running_loss / len(loader)
    
    auc_scores = []
    for i in range(all_labels.shape[1]):
        if len(np.unique(all_labels[:, i])) > 1:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            auc_scores.append(auc)
        else:
            auc_scores.append(0.0)
    
    mean_auc = np.mean(auc_scores)
    return epoch_loss, mean_auc, auc_scores

## Training Loop

In [ ]:
save_dir = MODELS_DIR / 'resnet50'
save_dir.mkdir(exist_ok=True)

history = {
    'train_loss': [],
    'val_loss': [],
    'val_auc': [],
    'lr': []
}

best_auc = 0.0
patience_counter = 0
early_stop_patience = 5

print(f"Starting training for {NUM_EPOCHS} epochs...\n")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 50)
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auc, per_class_auc = evaluate(model, val_loader, criterion, device)
    
    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['lr'].append(current_lr)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val AUROC: {val_auc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
            'per_class_auc': per_class_auc,
            'diseases': diseases
        }, save_dir / 'best_model.pth')
        
        print(f"New best model saved! AUROC: {best_auc:.4f}")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{early_stop_patience}")
    
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

total_time = time.time() - start_time

print(f"\nTraining completed!")
print(f"Best validation AUROC: {best_auc:.4f}")
print(f"Total training time: {total_time/3600:.2f} hours")

## Save Training History

In [ ]:
with open(save_dir / 'training_history.json', 'w') as f:
    json.dump({
        'model': 'ResNet50',
        'best_val_auc': float(best_auc),
        'total_epochs': len(history['train_loss']),
        'training_time_hours': total_time / 3600,
        'history': history
    }, f, indent=2)

print(f"Training history saved to {save_dir / 'training_history.json'}")

## Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_auc'], label='Val AUROC', marker='o', color='green')
ax2.axhline(y=best_auc, color='r', linestyle='--', label=f'Best ({best_auc:.4f})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUROC')
ax2.set_title('Validation AUROC')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training curves saved to {save_dir / 'training_curves.png'}")